[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tnecniv-Teikram/NSA-hyperspectral-course-2026/blob/main/exercises/ex_01_agriculture.ipynb)

*Running in Colab? Run the setup cell below first — it clones the course and installs what this notebook needs. Running locally? It does nothing.*

# 🌾 Exercises: Precision agriculture & crop intelligence
### Space Academy 2026 · Hyperspectral course · Week 2 — Project scoping and use case definition (Practical)

Go beyond the four broadband indices of the theme notebook and use what only narrow bands can offer: the red edge, the cellulose absorption and full-spectrum segmentation — on irrigated fields of the Konya plain, Türkiye's main grain basket.

These exercises accompany the theme notebook [`01_agriculture_crop_intelligence.ipynb`](https://github.com/Tnecniv-Teikram/813-hyperspectral-hackathon/blob/main/01_agriculture_crop_intelligence.ipynb). Run that notebook first
(or at least read it) — here we reuse its scene and indices, loaded with the course helper module `hsi_utils`.

> Work through the exercises in order — they get harder (⭐ → ⭐⭐⭐). Each has a hint; worked solutions are in `exercises/solutions/`.

**What you will practise**
- Turning an index threshold into an area estimate (hectares) and testing its sensitivity
- NDVI saturation and why red-edge indices help over dense canopies
- Computing the red-edge position from narrow bands
- Separating dry residue from bare soil with the Cellulose Absorption Index
- Unsupervised segmentation (PCA + k-means) of a full-spectrum window

**Data**
- `20250608_091605_90_4001` — Çumra, Konya plain, Türkiye (8 June 2025)
- `data/labels/…_worldcover2021_30m.tif` — ESA WorldCover 2021, aggregated to the Tanager 30 m grid

In [ ]:
# --- Setup: works both in Google Colab and in a local environment --------------------
# In Colab this clones the course repository, installs the geospatial packages that Colab
# does not ship, and moves into the course folder so the notebook finds hsi_utils and data/.
# Locally it does nothing at all — your own environment is already active.
import os
import pathlib
import subprocess
import sys

REPO = "Tnecniv-Teikram/NSA-hyperspectral-course-2026"          # change with: python tools/set_repo_url.py <owner>/<repo>

if "google.colab" in sys.modules:
    course = pathlib.Path("/content/hyperspectral-course")
    if not course.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        f"https://github.com/{REPO}.git", str(course)], check=True)
    os.chdir(course)
    sys.path.insert(0, str(course))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "h5py", "rasterio", "rioxarray", "xarray", "geopandas"], check=True)
    print("Colab ready — course at", course)
else:
    print("Local environment:", sys.executable.split("/")[-3])


## Setup

In [ ]:
# hsi_utils.py lives in the course folder (.. from here)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# On Google Colab: upload hsi_utils.py next to this notebook, change the line above to
# sys.path.insert(0, ".") and uncomment:
# %pip install -q h5py numpy scipy matplotlib scikit-learn pandas rasterio requests

import numpy as np
import matplotlib.pyplot as plt
import hsi_utils as hu

plt.rcParams["figure.dpi"] = 90

Load the scene and the bands used in the theme notebook. `hu.get_scene()` finds the local HDF5 or downloads it (~1 GB) from Planet's STAC catalogue.

In [ ]:
SCENE_ID = "20250608_091605_90_4001"   # Çumra, Konya plain, Türkiye (agriculture), 8 June 2025
scene = hu.get_scene(SCENE_ID)
print(scene.summary())

valid = scene.valid_mask()
R = {w: scene.read_band(w) for w in (570, 665, 705, 860, 1650)}   # the bands of notebook 01
NDVI = hu.nd(R[860], R[665])
NDRE = hu.nd(R[860], R[705])
PIXEL_HA = scene.res_x * scene.res_y / 10_000                     # 30 m × 30 m = 0.09 ha

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].imshow(scene.rgb()); axes[0].set_title("True colour"); axes[0].axis("off")
im = axes[1].imshow(NDVI, cmap="RdYlGn", vmin=-0.1, vmax=0.9); axes[1].set_title("NDVI"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()

## Exercise 1 — How many hectares are green?  ⭐

In early June most rain-fed cereals on the Konya plain are already drying, so fields that are still green are
very likely **irrigated**.

1. Count the valid pixels with NDVI > 0.5 and convert the count to hectares and km².
2. What percentage of the valid scene is that?
3. Repeat with thresholds 0.4 and 0.6. How sensitive is the area estimate to the threshold?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

One pixel is 30 m × 30 m = 900 m² = 0.09 ha (`PIXEL_HA`). Count with `(valid & (NDVI > t)).sum()`.

</details>

## Exercise 2 — NDVI saturates — does NDRE?  ⭐⭐

NDVI tends to *saturate* over dense canopies: once the leaf area index is high, it stops increasing. NDRE uses
the red-edge band (705 nm), which is absorbed less strongly by chlorophyll and so keeps responding.

1. Make a 2-D histogram of NDVI (x) against NDRE (y) for all valid pixels.
2. For pixels with NDVI > 0.8, compare the spread (standard deviation) of NDVI and of NDRE.
3. In which NDVI range does NDRE keep separating pixels that NDVI lumps together?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`plt.hist2d(NDVI[ok], NDRE[ok], bins=200, cmin=1, norm="log")`. For part 2, compare `np.nanstd` of both
indices within the mask — also compare the *relative* spread (std / range of the index over the scene).

</details>

## Exercise 3 — Red-edge position (REP) map  ⭐⭐

The **red-edge position** is the wavelength of the steepest slope between the red absorption and the NIR
plateau. It shifts to longer wavelengths as chlorophyll increases. With narrow bands you can estimate it with
the four-point linear interpolation of Guyot & Baret (1988):

$$R_{re} = \frac{R_{670} + R_{780}}{2}, \qquad
\mathrm{REP} = 700 + 40\,\frac{R_{re} - R_{700}}{R_{740} - R_{700}} \;\; [\mathrm{nm}]$$

1. Compute REP for vegetated pixels (NDVI > 0.5) and map it.
2. Plot its histogram and report the 5th, 50th and 95th percentiles.
3. Why can't you compute this with Sentinel-2 as precisely? (Look at `hu.SENTINEL2_BANDS`.)

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Read the four bands with `scene.read_band(670)`, etc. Set REP to NaN outside the vegetation mask with
`np.where(mask, rep, np.nan)`. A colour range of 710–735 nm works well.

</details>

## Exercise 4 — Dry residue vs bare soil: the Cellulose Absorption Index  ⭐⭐

Senescent steppe grass, stubble and bare soil all have **low NDVI** — NDVI cannot tell them apart. Dry plant
material, however, has a cellulose/lignin absorption near 2100 nm that soils lack. The **Cellulose
Absorption Index** (Nagler et al., 2000) measures it:

$$\mathrm{CAI} = 0.5\,(R_{2031} + R_{2211}) - R_{2101}$$

1. Compute CAI for the scene.
2. Compare CAI and NDVI for three groups: green crops (NDVI > 0.6), bare soil (NDVI < 0.12) and dry steppe
   (WorldCover class 30 = grassland, 0.12 < NDVI < 0.25). Use box plots.
3. Which index separates bare soil from dry vegetation better?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

WorldCover labels aligned to this scene: `lab, purity = hu.load_labels(SCENE_ID)`. For box plots:
`ax.boxplot([values_a, values_b, values_c], showfliers=False)` then `ax.set_xticks([1, 2, 3], names)`.

</details>

## Exercise 5 — Segment the strip fields with k-means  ⭐⭐⭐

The striped irrigated fields south-east of the scene centre are a nice test for **unsupervised segmentation**.

1. Read the full-band cube for the window `(300, 500, 400, 600)` (rows 300–500, cols 400–600).
2. Reduce it to 10 principal components (PCA) and cluster the pixels with k-means (k = 6).
3. Show the true-colour window, NDVI and the cluster map side by side, and plot the mean spectrum of each
   cluster.
4. Give each cluster a name based on its mean spectrum and report its area in hectares.

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`cube, wl = scene.read_cube(window)` → `X, ok = hu.cube_to_pixels(cube)` gives an (n_pixels × bands) matrix.
Use `sklearn.decomposition.PCA(10).fit_transform(X)` and `sklearn.cluster.KMeans(6, n_init=10, random_state=0)`.
Put the labels back on the grid with `hu.pixels_to_image(labels, ok)`.

</details>

## Exercise 6 — Reflection — stress, or just a different crop?  ⭐

The theme notebook says hyperspectral data can detect crop stress "weeks before visible damage". Using
your REP and NDRE results, answer in a short markdown cell:

1. Two neighbouring green fields have the same NDVI but one has a clearly lower REP. List at least three
   explanations other than "stress".
2. What additional data would you need to turn the REP map into a **stress alert** that a farmer or an
   agricultural ministry could act on?

In [ ]:
# (optional) any numbers you want to quote — then answer in a markdown cell

<details><summary>💡 Hint</summary>

Think about crop type, sowing date / growth stage, irrigation schedule, soil background, and what "normal"
looks like for each field over time.

</details>

## Key takeaways

- An area statistic is only as good as its threshold — test the sensitivity and report it.
- Red-edge indices and the REP keep responding where NDVI saturates.
- SWIR features (such as CAI at 2100 nm) separate materials that are identical in NDVI.
- Unsupervised clusters show spectral structure; labels are needed to turn them into classes.
- A stress product needs field-level baselines, time series and ground truth.

---

> 📖 Data: Tanager — CC-BY-4.0 © Planet Labs PBC  
> 📖 Labels: ESA WorldCover 2021 v200 — CC-BY 4.0